# Observing Teams with `run_stream`

## What is streaming?
`team.run(...)` waits silently for the *whole* conversation to finish, then hands back one big result. **`team.run_stream(...)` is different — it gives you each event the *moment* it happens**, like a live ticker. You don't wait. You watch.

Concretely, `run_stream` returns an **async iterator**. You loop over it with `async for`, and on every iteration you get the next event:
- A message from one of the agents, *as soon as that agent finishes speaking*.
- Then the next agent's message when they finish.
- ...and so on, in real time.
- The **very last** item is a special `TaskResult` — the signal the team is done.

## Why streaming is the right tool for observing teams
Teams can take a while (multiple LLM calls, multiple speakers). Streaming lets you:
- **Print live** — show progress instead of a frozen screen.
- **Filter on the fly** — drop events you don't care about as they arrive.
- **Tally as you go** — add up tokens per agent without waiting for the end.

## What we'll do
1. Live view with `Console` (consumes the stream for you)
2. Custom `async for` loop (consume the stream yourself)
3. Token usage per agent, tallied while streaming

We'll use a small writer/critic team.

## 1. Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

writer = AssistantAgent(
    name="writer",
    model_client=model_client,
    system_message="Write ONE 3-sentence paragraph on the topic.",
)

critic = AssistantAgent(
    name="critic",
    model_client=model_client,
    system_message="If the paragraph is clear, reply APPROVE. Else give one suggestion.",
)

team = RoundRobinGroupChat(
    participants=[writer, critic],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(6),
)

## 2. Live view with `Console`

In [ ]:
from autogen_agentchat.ui import Console

await Console(team.run_stream(task="Explain what an AI agent is."))

## 3. Custom loop

Iterate the stream yourself. The last item is a `TaskResult` — that's the done signal.

In [ ]:
from autogen_agentchat.base import TaskResult

await team.reset()

async for item in team.run_stream(task="Explain what RAG is."):
    if isinstance(item, TaskResult):
        print(f"[done] {item.stop_reason}")
    else:
        print(f"[{item.source}] {str(item.content)[:60]}")

## 4. Token usage per agent

Each LLM event has `models_usage`. Aggregate it by `source`.

In [ ]:
from collections import defaultdict

await team.reset()
usage = defaultdict(int)

async for item in team.run_stream(task="Explain what fine-tuning is."):
    u = getattr(item, "models_usage", None)
    if u is not None:
        usage[item.source] += u.prompt_tokens + u.completion_tokens

for name, total in usage.items():
    print(f"{name}: {total} tokens")

## 5. Clean up

In [ ]:
await model_client.close()